# Exploratory Data Analysis (EDA) - Sales Prediction System

This notebook contains a professional exploratory data analysis (EDA) of the `Advertising.csv` dataset, which records advertising spends in three media channels (TV, Radio, Newspaper) and corresponding Sales volumes. 

Our goal is to understand the distributions, relations, and potential data cleaning requirements (e.g., missing value imputation, outlier capping) before feeding the features into the training pipeline.

### 1. Environment Setup and Data Loading
We will load necessary scientific and visualization packages and load the dataset from the standardized `data/` subdirectory.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetic
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load dataset
data_path = "../data/Advertising.csv"
df = pd.read_csv(data_path)

# Clean up index column if present
unnamed_cols = [col for col in df.columns if col.startswith("Unnamed:") or col == ""]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)

print(f"Dataset successfully loaded. Shape: {df.shape}")

### 2. Dataset Overview
Let's display the first few rows of the data, inspect columns data types, and view the summary statistics.

In [ ]:
print("--- First 5 Rows ---")
display(df.head())

print("\n--- Columns Info ---")
display(df.info())

print("\n--- Summary Statistics ---")
display(df.describe())

### 3. Null Analysis
We will inspect if there are any missing values in the dataset. Missing values must be handled by imputation.

In [ ]:
missing_values = df.isnull().sum()
print("Missing value count per column:")
print(missing_values)

if missing_values.sum() == 0:
    print("\nNo missing values detected in the dataset.")
else:
    print(f"\nDetected {missing_values.sum()} missing values. Imputation required.")

### 4. Univariate Distributions
Let's visualize the probability density distributions and histograms for our features (`TV`, `Radio`, `Newspaper`) and target (`Sales`).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

sns.histplot(df['TV'], kde=True, ax=axes[0, 0], color='blue')
axes[0, 0].set_title('TV Ad Spend Distribution')

sns.histplot(df['Radio'], kde=True, ax=axes[0, 1], color='green')
axes[0, 1].set_title('Radio Ad Spend Distribution')

sns.histplot(df['Newspaper'], kde=True, ax=axes[1, 0], color='orange')
axes[1, 0].set_title('Newspaper Ad Spend Distribution')

sns.histplot(df['Sales'], kde=True, ax=axes[1, 1], color='red')
axes[1, 1].set_title('Sales Target Distribution')

plt.tight_layout()
plt.show()

**Key Observations:**
- `TV` and `Radio` spends are relatively uniform / spread out across their range.
- `Newspaper` spend is right-skewed; most spends are clustered at lower amounts, with a long right tail.
- `Sales` is normally distributed, which is an ideal characteristic for linear regression model assumptions.

### 5. Outlier Visualization
We will plot boxplots to detect any outliers in the features. Outliers can adversely affect linear models and should be capped using methods like the IQR method.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df.drop(columns=['Sales']))
plt.title('Outliers Identification Boxplot')
plt.ylabel('Ad Spend (USD)')
plt.show()

**Key Observations:**
- There are no visible outliers in `TV` or `Radio` spends.
- `Newspaper` spend has a few outlier points above 100. Capping values using the 1.5 * IQR multiplier will help standardise this column.

### 6. Bivariate Correlation Analysis
Let's look at the correlation matrix and plot a heatmap to analyze the relationships between our variables.

In [ ]:
corr = df.corr()
print("Correlation Matrix:")
display(corr)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap')
plt.show()

**Key Observations:**
- `TV` has a very high positive correlation with `Sales` (0.78), indicating it is likely the most predictive feature.
- `Radio` also has a strong positive correlation with `Sales` (0.58).
- `Newspaper` has the lowest correlation with `Sales` (0.23).
- Features do not display extreme multicollinearity with each other (all pairwise correlations between TV, Radio, and Newspaper are < 0.40), indicating dropping them based on multicollinearity is not required unless our interaction features introduce high collinearity.

### 7. Pairwise Relationships (Scatter Matrix)
We can visualize the pairwise relationships using `pairplot`. This helps us examine if any relationship is non-linear (which would justify engineering interaction or polynomial features).

In [ ]:
sns.pairplot(df, diag_kind='kde', plot_kws={'alpha': 0.7, 'edgecolor': 'k', 'linewidth': 0.5})
plt.suptitle('Pairplot of Advertising Dataset Features and Target', y=1.02)
plt.show()

### 8. Conclusion and Feature Engineering Rationale

1. **TV and Radio Interactions**: The scatter plots of `TV` vs `Sales` and `Radio` vs `Sales` show positive trends, but their combined effect is often non-linear (synergistic advertising effect). Creating a `TV_Radio_interaction` feature (TV * Radio) is highly motivated.
2. **Total Spend**: Aggregating all channels into `total_ad_spend` will help capture overall budget size correlation with sales.
3. **Scaling**: The range of features varies (TV goes up to 300, Newspaper up to 110, Radio up to 50). Distance-based/regularized models will benefit significantly from StandardScaler / MinMaxScaler.